# Data Injestion with LakeFlow Connect

Listing files in our volume

In [0]:
SELECT 1;

In [0]:
LIST '/Volumes/workspace/default/raw_files'

### Creating Single Delta Table from raw files 

In [0]:
CREATE OR REPLACE TABLE customers_bronze AS 
SELECT *
FROM read_files(
  '/Volumes/workspace/default/raw_files/',
  format => 'csv',
  header => true
);

### Setting up Production level streaming delta table with 1 Hour Schedule time

In [0]:
SELECT * FROM customers_bronze


In [0]:
CREATE OR REFRESH STREAMING TABLE customers_bronze_streaming
SCHEDULE EVERY 1 HOUR
AS
SELECT *
FROM STREAM read_files(
  '/Volumes/workspace/default/raw_files/',
  format => 'csv',
  header => true
);

In [0]:
DESCRIBE TABLE EXTENDED customers_bronze_streaming

In [0]:
DESCRIBE HISTORY customers_bronze_streaming

Refreshing Column After adding new files to folder.

In [0]:
REFRESH STREAMING TABLE customers_bronze_streaming

In [0]:
SELECT * FROM customers_bronze_streaming

In [0]:
DESCRIBE HISTORY customers_bronze_streaming

Result of running the DESCRIBE HISTORY AFTER AN HOUR to see real demo.

In [0]:
SELECT * FROM customers_bronze_streaming

In [0]:
DESCRIBE HISTORY customers_bronze_history

### Injestion Data with Meta Data

In [0]:
SELECT
*, 
-- Last data source file modification time
_metadata.file_name AS source_file,
-- Ingest data source file name
current_timestamp() as ingestion_time
-- Ingestion timestamp
FROM read_files (
"/Volumes/workspace/default/raw_files");

It's working as expected , so lets create final bronze table from this. 

In [0]:
DROP TABLE IF EXISTS customers_bronze_with_meta_data;

CREATE TABLE customers_bronze_with_meta_data AS
SELECT
*, 
-- Last data source file modification time
_metadata.file_name AS source_file,
-- Ingest data source file name
current_timestamp() as ingestion_time
-- Ingestion timestamp
FROM read_files (
"/Volumes/workspace/default/raw_files");

SELECT * FROM customers_bronze_with_meta_data;

In [0]:
SELECT 
source_file,
COUNT(*) AS total
FROM customers_bronze_with_meta_data
GROUP BY source_file
ORDER BY source_file;

### Working with JSON Files

In [0]:
LIST '/Volumes/workspace/default/json_files'

In [0]:
FROM read_files(
  '/Volumes/workspace/default/json_files',
  format => 'json',
  multiline => true
);

In [0]:
CREATE OR REPLACE TABLE users_bronze_raw
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/default/json_files',
  format => 'json',
  multiline => true
);

In [0]:
SELECT
  explode(users) AS user
FROM users_bronze_raw;

copied one users array item to ofer schema using STRUCT

In [0]:
SELECT schema_of_json('{"createdAt":"2023-03-15T09:22:00Z","email":"alice@example.com","id":1,"lastActive":"2024-01-20T14:30:00Z","preferences":{"language":"en","notifications":true,"theme":"light"},"profile":{"avatar":"https://example.com/avatars/alice.jpg","bio":"Curious explorer of digital realms","firstName":"Alice","lastName":"Wonderland","location":"San Francisco, CA","website":"https://alice-wonder.dev"},"stats":{"followers":1250,"following":89,"likes":3420,"posts":156},"username":"alice_wonder"}') AS schema

In [0]:
CREATE OR REPLACE TABLE users_silver AS
SELECT
  explode(users) AS user
FROM users_bronze_raw;

In [0]:
CREATE OR REPLACE TABLE users_silver_flat AS
SELECT
  user.id,
  user.username,
  user.email,
  user.createdAt,
  user.lastActive,

  user.profile.firstName,
  user.profile.lastName,
  user.profile.location,

  user.preferences.language,
  user.preferences.notifications,
  user.preferences.theme,

  user.stats.followers,
  user.stats.following,
  user.stats.likes,
  user.stats.posts
FROM users_silver;

In [0]:
SELECT * FROM users_silver_flat;

In [0]:
CREATE OR REPLACE TABLE users_gold AS
SELECT
  user.id,
  user.username,

  user.profile.location AS location,
  user.profile.firstName,
  user.profile.lastName,

  user.stats.followers,
  user.stats.following,
  user.stats.likes,
  user.stats.posts,

  CASE 
    WHEN user.stats.followers > 1000 THEN 'High'
    WHEN user.stats.followers > 100 THEN 'Medium'
    ELSE 'Low'
  END AS user_tier,

  CASE 
    WHEN user.lastActive IS NOT NULL THEN 1 ELSE 0
  END AS is_active
FROM users_silver;

In [0]:
SELECT id, username, user_tier FROM users_gold;